# 环节 02 · Embedding 查表（配套 Notebook）

> 配套长文：[环节02-Embedding查表详解.md](./环节02-Embedding查表详解.md)
> 定位：把长文里"编号 → 向量"的机制跑成可复现的代码。全部**纯 Python 标准库**实现（不用 numpy / torch）。

**怎么跑**

- 依赖：无。逐格 `Shift+Enter`；后面的格子依赖前面已执行的变量。
- 想换语料：只改 §5 的 `SENTS`（改完重跑 §5 三格即可）。

**地图：本 Notebook ↔ 长文章节**

| 本 Notebook | 长文章节 | 验证什么 |
|---|---|---|
| §1 编号的三个假象 | §1 | 为什么整数 id 不能直接喂给网络 |
| §2 词义坐标表 | §2.1 | 查表 = 取一行；猫狗近 / 猫车远 |
| §3 查表 ≡ one-hot × W_E | §2.5 | 两种视角严格等价 + one-hot 的两个死穴 |
| §4 梯度只回传给"命中"的行 | §2.3 / §2.3.1 | scatter_add 机制、低频词为什么学不动 |
| §5 亲手把"猫狗"训近 | §2.1 / §2.2 | 随机乱点 → 学出语义近邻（全流程自己跑一遍） |
| §6 尺度与参数账 | §2.3.2 / §2.3.3 / §3 | 初始化 0.02、\|V\|×d、weight tying |


## 1. 为什么编号不能直接用（长文 §1）

上一站 [环节 01](./环节01-Tokenizer分词详解.md) 交出来的是一串整数 id。id 是**离散、无语义**的，而且自带三个"人为假象"：

| 坑 | 例子 | 为什么致命 |
|---|---|---|
| ① 编号是假的 | `7 + 35 = 42` | 模型爱做加法，但"爱 + 猫 = 42"没有意义 |
| ② 大小是假的 | `35 > 7` | 不代表"猫"比"爱"高级 |
| ③ 远近是假的 | 猫 = 35、狗 = 36 挨着 | 但"爱"= 7 离它们很远，编号顺序与意思无关 |

跑一遍看这三个坑到底假在哪：


In [ ]:
ids = {"我": 2, "爱": 7, "猫": 35, "狗": 36, "汽车": 200}

print("① 加法没有意义：")
print(f"   id(爱) + id(猫) = {ids['爱']} + {ids['猫']} = {ids['爱'] + ids['猫']}"
      "   → 42 不对应任何词")

print("\n② 大小没有意义：")
print(f"   id(猫)={ids['猫']} > id(爱)={ids['爱']}，但“猫”并不比“爱”高级")

print("\n③ 远近没有意义：")
print(f"   |id(猫) - id(狗)|   = {abs(ids['猫'] - ids['狗'])}"
      "   ← 挨着（这次碰巧语义也近）")
print(f"   |id(猫) - id(汽车)| = {abs(ids['猫'] - ids['汽车'])}"
      "   ← 隔很远，但“猫”和“汽车”确实无关……编号纯属巧合")

print("\n结论：编号只提供“索引”，不携带语义 → 需要一个“意思相近就靠得近”的表示。")


## 2. 词义坐标表：查表就是"取一行"（长文 §2.1）

给每个 id 配一条稠密向量（几百~几千维），**按行号取一行**即可——查表瞬间完成，不涉及任何计算。

先用长文 §2.1 的那张 2 维示意表（维度降到 2 只是为了能画在纸上）：


In [ ]:
import math


def dist(p, q):
    """两个坐标点的欧氏距离。"""
    return math.sqrt(sum((x - y) ** 2 for x, y in zip(p, q)))


# 词义坐标表（示意，2 个数 = 2 维）；真实模型是 |V| × 4096
EMB = {
    "我":   (0.2, 0.9),
    "爱":   (0.1, 0.6),
    "猫":   (1.0, 0.5),
    "狗":   (0.9, 0.7),
    "鱼":   (0.8, 1.0),
    "汽车": (-0.9, 0.1),
}

print("“我爱猫” → 查三行 →", [EMB[w] for w in "我爱猫"])
print()
print(f"猫 ↔ 狗  ： {dist(EMB['猫'], EMB['狗']):.2f}   （很近）")
print(f"猫 ↔ 汽车： {dist(EMB['猫'], EMB['汽车']):.2f}   （很远）")
print("\n注意：这张表不是人写的，是训练时被“挪”出来的 —— §5 会亲手训一次。")


In [ ]:
def scatter(title, points, width=44, height=12):
    """把 2 维向量画成字符散点图（字母标记；多个词落在同一格时显示 *）。"""
    names = list(points)
    xs = [points[n][0] for n in names]
    ys = [points[n][1] for n in names]
    x0, x1 = min(xs) - 0.2, max(xs) + 0.2
    y0, y1 = min(ys) - 0.2, max(ys) + 0.2

    buckets = {}
    for i, name in enumerate(names):
        x, y = points[name]
        col = int((x - x0) / (x1 - x0) * (width - 1))
        row = int((y1 - y) / (y1 - y0) * (height - 1))
        buckets.setdefault((row, col), []).append(i)

    grid = [[" "] * width for _ in range(height)]
    for (row, col), idxs in buckets.items():
        grid[row][col] = chr(ord("A") + idxs[0]) if len(idxs) == 1 else "*"

    print(f"\n{title}")
    for r in range(height):
        print("  |" + "".join(grid[r]))
    print("  +" + "-" * width + "→ x")
    print("   " + "  ".join(f"{chr(ord('A') + i)}={n}" for i, n in enumerate(names)))
    if any(len(v) > 1 for v in buckets.values()):
        print("   （* = 几个词完全重叠在同一个坐标上）")


scatter("词义地图（长文 §2.1 的坐标）", EMB)
print("\n猫(C) 和狗(D) 挤在一起，汽车(F) 远在另一头 —— 这就是“意思近的住得近”。")


## 3. 查表 ≡ one-hot × W_E（长文 §2.5）

前向只有一句 `x = W_E[token_id]`，看着"没做运算"，所以反向传播的第一反应是"梯度往哪传"。答案在长文 §2.5：**"取第 i 行"可以写成一次标准矩阵乘** `x = e_i · W_E`（`e_i` 是第 i 位为 1 的 one-hot 行向量），于是整条链自然可导。

先用代码验证这两种视角**严格等价**：


In [ ]:
W_E = [                       # 4 行 × 3 列的玩具 Embedding（真实是 130k × 4096）
    [0.2, 0.9, 0.1],          # id 0
    [0.1, 0.6, 0.3],          # id 1
    [1.0, 0.5, 0.2],          # id 2
    [-0.9, 0.1, 0.4],         # id 3
]


def one_hot(i, n):
    """第 i 位为 1、其余为 0 的 one-hot 向量。"""
    return [1.0 if k == i else 0.0 for k in range(n)]


def vec_mat(vec, matrix):
    """行向量 × 矩阵：out[j] = Σ_i vec[i] * matrix[i][j]。

    注意方向：e_i 是 |V| 维行向量、W_E 是 |V| × d，乘出来才是 d 维的一行。
    """
    return [sum(vec[i] * matrix[i][j] for i in range(len(vec)))
            for j in range(len(matrix[0]))]


for i in (2, 3):
    via_one_hot = vec_mat(one_hot(i, len(W_E)), W_E)     # e_i · W_E
    via_lookup = W_E[i]                                  # W_E[i]
    same = all(abs(a - b) < 1e-12 for a, b in zip(via_one_hot, via_lookup))
    print(f"id={i}: one-hot × W_E = {[round(x, 2) for x in via_one_hot]}"
          f"   直接取第 {i} 行 = {[round(x, 2) for x in via_lookup]}   严格相等 = {same}")

print("\n→ 所以“查表”不是不可导的魔法，它就是一次矩阵乘；")
print("  环节01 交出来的整数 id，本质就是 one-hot 的“压缩存储格式”。")


In [ ]:
# 那为什么不直接用 one-hot？两个死穴：① 没有相似度 ② 维度爆炸
V, d = 130_000, 4096          # 真实量级：词表 13 万、隐藏维度 4096

a, b = one_hot(5, V), one_hot(6, V)
inner = sum(x * y for x, y in zip(a, b))
d12 = math.sqrt(sum((x - y) ** 2 for x, y in zip(a, b)))

print("死穴①：任意两个 one-hot 的关系恒定，与语义无关")
print(f"   内积 = {inner}    距离 = {d12:.4f}（恒为 √2）")
print("   → 猫和狗、猫和汽车，在 one-hot 里一样远 —— 相似度为零")

print("\n死穴②：维度")
print(f"   one-hot  = {V:,} 维/词    Embedding = {d:,} 维/词"
      f"    → 稠密表示省 {V / d:.0f} 倍")
print(f"   而且 one-hot 里 {(V - 1) / V:.4%} 的位置是 0，纯浪费带宽与显存 → 工程上从不物化它")


## 4. 梯度只回传给"命中"的那一行（长文 §2.3）

前向 `x_i = e_i · W_E` 之后，反向传播一路从输出头传回 `∂L/∂x`，到这一层按链式法则：

```
∂L/∂W_E = Σ_i e_iᵀ ⊗ (∂L/∂x_i)      # 只对 batch 里出现过的 token 行非零
```

**一句大白话：被取中的行收到下游原样送回的一笔梯度，没被取中的行本轮梯度为 0。** 实现上就是一个 `scatter_add`。


In [ ]:
W = [[0.10 * i, 0.20 * i] for i in range(4)]        # 4 个词 × 2 维的玩具表
grad_x = {0: (1.0, -0.5), 2: (0.3, 0.8)}           # 下游反传回来的 ∂L/∂x：本轮只有 id 0 和 id 2 出现过

lr = 0.1
before = [row[:] for row in W]
for i, g in grad_x.items():                        # 只遍历“命中”的行 = scatter_add
    for k in range(len(g)):
        W[i][k] -= lr * g[k]

print("词行    更新前            更新后            本轮状态")
print("-" * 62)
for i in range(len(W)):
    tag = "命中 → 收到梯度，挪一步" if i in grad_x else "未命中 → 梯度为 0，原地不动"
    old = str([round(v, 2) for v in before[i]])
    new = str([round(v, 2) for v in W[i]])
    print(f"id {i}   {old:<16} {new:<16} {tag}")

print("\n→ 本轮 batch 里没有 id 1 / id 3，所以它们这一行“纹丝没动”。")
print("  训练一条样本 = 只更新命中的那几行 —— 这就是 Embedding 层的更新机制。")


In [ ]:
# 推论（长文 §2.3.1）：一个词被更新的次数 = 它在语料里出现的次数
# 真实语料的词频是重尾（Zipf）分布 → 高频词被推上万次，长尾词可能只被推 1 次
import random
from collections import Counter

random.seed(3)
COMMON = "的 了 是 在 和 我 有 就 不".split()          # 9 个高频词
RARE = [f"长尾词{i}" for i in range(300)]              # 300 个长尾词

tokens = [random.choice(COMMON) if random.random() < 0.75 else random.choice(RARE)
          for _ in range(3000)]
freq = Counter(tokens)
counts = sorted(freq.values(), reverse=True)
once = sum(1 for c in counts if c == 1)
few = sum(1 for c in counts if c <= 2)

print(f"语料 {len(tokens)} 个 token；词表 {len(COMMON) + len(RARE)} 个词，"
      f"其中 {len(freq)} 个出现过")
print(f"top5 出现次数：{counts[:5]}")
print(f"只出现 1 次的词：{once} 个（{once / len(freq):.0%}）")
print(f"出现 ≤2 次的词  ：{few} 个（{few / len(freq):.0%}）")
print(f"最高频 : 最低频 = {counts[0]} : {counts[-1]} → 累计位移差 {counts[0]} 倍")

print("\n→ 这些长尾行几乎停在初始小随机值附近，等于“占了名额但学不动”的死 token。")
print("  对照长文 §6.1（环节01）的词表大小专题：词表一味变大，新增的多是这类行。")


## 5. 亲手把"猫狗"训近（长文 §2.1 / §2.2）

长文说："一开始整张表是随机乱点，读文章发现猫和狗总出现在相似句子里，就一点点把它们挪近。"

下面把这条链路完整跑一遍，只用 12 个词的玩具语料：

```
语料 → 共现统计 → PPMI 加权 → 矩阵分解（梯度下降） → 语义向量
```

> 说明：真实 LLM **不用 PPMI**，而是用 next-token 预测 + 反向传播端到端学（长文 §2.3）。
> 但"该不该近"这件事，两组方法都由**共现/上下文**决定，所以这里用 PPMI 这条更小、更确定的路径演示（Word2Vec / GloVe 那一路的血缘）。


In [ ]:
SENTS = [
    "猫 很 可爱",
    "狗 很 可爱",
    "猫 在 睡觉",
    "狗 在 睡觉",
    "我 爱 猫",
    "我 爱 狗",
    "我 爱 鱼",
    "汽车 在 跑",
    "我 开 汽车",
]

WORDS = sorted({w for s in SENTS for w in s.split()})
W2I = {w: i for i, w in enumerate(WORDS)}
n = len(WORDS)

# 共现矩阵：窗口 = 1，左右邻居都记
C = [[0] * n for _ in range(n)]
for sent in SENTS:
    ws = sent.split()
    for a, w in enumerate(ws):
        for b in (a - 1, a + 1):
            if 0 <= b < len(ws):
                C[W2I[w]][W2I[ws[b]]] += 1

print(f"词表（{n} 个）：{WORDS}\n")
print("共现矩阵（行 = 词，列 = 它的邻居；数字是“一起出现过几次”）")
print("      " + " ".join(f"{w:>4}" for w in WORDS))
for i, w in enumerate(WORDS):
    print(f"{w:>4}  " + " ".join(f"{C[i][j]:>4}" for j in range(n)))

print("\n看“猫”和“狗”两行：完全一样（很/在/爱 都是 1）；")
print("再看“汽车”行：(在,1) (跑,1) (开,1) —— 和猫狗几乎不重叠。语义差别已经在计数里了。")


In [ ]:
total = sum(sum(r) for r in C)
marginal = [sum(r) for r in C]


def ppmi(i, j):
    """PPMI：共现得比“两者各自随机碰面”多多少（取正部）。

    纯共现次数会被高频词带偏（“的”和谁都共现多），PPMI 折算掉词频，只留下“真关联”。
    """
    if C[i][j] == 0:
        return 0.0
    p_ij = C[i][j] / total
    p_i = marginal[i] / total
    p_j = marginal[j] / total
    return max(0.0, math.log(p_ij / (p_i * p_j)))


P = [[ppmi(i, j) for j in range(n)] for i in range(n)]

print("PPMI 矩阵的“猫”行与“汽车”行（0 = 从没一起出现）")
print("      " + " ".join(f"{w:>5}" for w in WORDS))
for name in ("猫", "汽车"):
    i = W2I[name]
    print(f"{name:>4}  " + " ".join(f"{P[i][j]:>5.2f}" for j in range(n)))

same_cat_dog = all(abs(P[W2I["猫"]][j] - P[W2I["狗"]][j]) < 1e-12 for j in range(n))
print(f"\n“猫”行与“狗”行完全相同 = {same_cat_dog}   ← 它俩的上下文一模一样")


In [ ]:
def dot(a, b):
    return sum(x * y for x, y in zip(a, b))


def norm(a):
    return math.sqrt(sum(x * x for x in a))


def cos(a, b):
    na, nb = norm(a), norm(b)
    return dot(a, b) / (na * nb) if na and nb else 0.0


def train_embeddings(d=2, steps=1500, lr=0.02, seed=0):
    """把 PPMI 矩阵分解成 d 维向量：让 v_i · u_j 尽量拟合 PPMI(i,j)。

    初始是随机乱点，梯度下降让它按“谁该和谁近”逐渐摆好 —— 这就是 §2.2 的大白话版本。
    """
    random.seed(seed)
    U = [[random.gauss(0, 0.1) for _ in range(d)] for _ in range(n)]
    V = [[random.gauss(0, 0.1) for _ in range(d)] for _ in range(n)]
    start = [[U[i][k] + V[i][k] for k in range(d)] for i in range(n)]

    for _ in range(steps):
        for i in range(n):
            for j in range(n):
                if C[i][j] == 0:                 # 只拟合“真共现过”的词对
                    continue
                err = dot(U[i], V[j]) - P[i][j]
                gu = [err * x for x in V[j]]     # 用旧值算两个梯度，再一起更新
                gv = [err * x for x in U[i]]
                for k in range(d):
                    U[i][k] -= lr * gu[k]
                    V[j][k] -= lr * gv[k]

    emb = [[U[i][k] + V[i][k] for k in range(d)] for i in range(n)]
    return emb, start


EMB_TRAINED, EMB_RANDOM = train_embeddings()

cat, dog, car = W2I["猫"], W2I["狗"], W2I["汽车"]
print("猫 ↔ 狗    猫 ↔ 汽车")
print(f"训练前（随机乱点）: {dist(EMB_RANDOM[cat], EMB_RANDOM[dog]):5.2f}"
      f"      {dist(EMB_RANDOM[cat], EMB_RANDOM[car]):5.2f}")
print(f"训练后（学出来的）: {dist(EMB_TRAINED[cat], EMB_TRAINED[dog]):5.2f}"
      f"      {dist(EMB_TRAINED[cat], EMB_TRAINED[car]):5.2f}")
print("→ 训练前猫和汽车甚至比猫和狗更近（纯随机）；训练后猫狗贴到一起，汽车被推远。\n")

print("“猫”的余弦相似度排行：")
rank = sorted(WORDS, key=lambda w: -cos(EMB_TRAINED[cat], EMB_TRAINED[W2I[w]]))
for w in rank:
    print(f"  {w:>4}: {cos(EMB_TRAINED[cat], EMB_TRAINED[W2I[w]]):+.3f}")
print("\n没有人在表里写“猫和狗近” —— 这是“上下文相似 → 向量被动挨近”的自然结果。")
print("注意：只有 2 维，相似度会普遍偏高（大家都挤在一起），**看排序不看绝对值**；")
print("      “猫”和“狗”余弦 = 1.000，是因为它俩的共现行完全一样（语料里完全平行）。")


In [ ]:
TR = {w: tuple(EMB_TRAINED[W2I[w]]) for w in WORDS}
RD = {w: tuple(EMB_RANDOM[W2I[w]]) for w in WORDS}

scatter("训练前：随机乱点（谁和谁近纯属偶然）", RD)
scatter("训练后：猫狗贴在一起，汽车在另一侧", TR)
print("\n注意这张图只有 2 维（真实模型 4096 维，画不出来）；维度越低越挤，是正常现象。")


## 6. 初始化尺度与参数账（长文 §2.3.2 / §2.3.3 / §3）

**初始化尺度**：现代 LLM 用标准差约 `0.02` 的小随机。太大 → 进门就盖过位置编码与后续层；太小 → 所有词挤在原点，要先"从零长出来"。


In [ ]:
def mean_pairwise_dist(d, std, count=20, seed=0):
    """随机初始化后，任意两个词向量的平均距离 —— 衡量“初始区分度”。"""
    random.seed(seed)
    vecs = [[random.gauss(0, std) for _ in range(d)] for _ in range(count)]
    ds = [math.sqrt(sum((x - y) ** 2 for x, y in zip(vecs[i], vecs[j])))
          for i in range(count) for j in range(i + 1, count)]
    return sum(ds) / len(ds)


d_model = 4096
print(f"维度 d = {d_model} 时，不同初始化标准差下的初始平均两两距离：")
for std in (0.001, 0.02, 0.5):
    note = {0.001: "太小 → 所有词几乎重合，要先从零长出来",
            0.02: "现代 LLM 常用（差异够用又不喧宾夺主）",
            0.5: "太大 → 进门就盖过后续层信号"}[std]
    print(f"  std = {std:<6} 平均距离 ≈ {mean_pairwise_dist(d_model, std):6.2f}   {note}")


In [ ]:
# 参数账（长文 §3 工程注记）
V, d_model = 130_000, 4096
one = V * d_model

print(f"Embedding 表 : {V:,} × {d_model:,} = {one / 1e8:.2f} 亿参数（输入查表）")
print(f"LM Head      : {V:,} × {d_model:,} = {one / 1e8:.2f} 亿参数（输出猜词，见环节08）")
print(f"不共享合计   : {2 * one / 1e8:.2f} 亿")
print(f"weight tying : {one / 1e8:.2f} 亿（两处复用同一张表，省掉一半）")

print("\n另外两点（长文 §2.3.3）：")
print("  · 分布式训练要 allreduce 整张 W_E 梯度 —— 梯度“稀疏”（只有命中行非零），通信却“稠密”（整张都要同步）；")
print("  · 所以超大规模常做词表并行（按行分片到各卡）+ tie embedding 省参数。")
print("  · LoRA 等冻结 base 的方案里，这一层不更新。")


## 7. 自测与面试自查（长文 §4 / §5）

| 问题 | 本 Notebook 的现场证据 |
|---|---|
| 为什么不能把整数编号直接喂给模型？ | §1：加法 / 大小 / 远近三个假象 |
| Embedding 表长什么样？ | §2：`\|V\| × d`，`向量 = W_E[token_id]` 取一行 |
| 猫狗为什么会近？谁规定的？ | §5：共现 → PPMI → 分解，全程无人标注 |
| 查表"没做运算"，梯度往哪传？ | §3：`e_i · W_E`，两种视角严格等价 |
| 梯度更新有什么特殊？ | §4：只回传命中的行（scatter_add），未命中为 0 |
| 低频词为什么学不动？ | §4 第二个实验：更新次数 = 出现次数，长尾只被推 1 次 |
| 为什么不用 Word2Vec 初始化？ | §5 用 PPMI 演示了静态共现那条线；大模型走端到端，静态词义会被后续层淹没 |
| 参数量多少？ | §6：`\|V\| × d` ≈ 5.3 亿，tie embedding 省一半 |

**下一站**：[环节 03 · 位置编码](./环节03-位置编码详解.md)——向量只回答了"这个字是谁"，还没说它"排第几"。
